## Deteccao e Contagem de Embalagens de Alimentos com YOLOv8

### Objetivo

Treinar um modelo de deteccao de objetos capaz de identificar e contar embalagens de alimentos em tres categorias: **arroz**, **feijao** e **outros**.

### Metodologia

1. Importar bibliotecas e definir configuracoes
2. Gerar dataset aumentado a partir de imagens base (data augmentation com OpenCV)
3. Dividir dataset em treino e validacao
4. Gerar labels no formato YOLO
5. Treinar modelo YOLOv8n (transfer learning)
6. Avaliar metricas (mAP, precision, recall)
7. Inferencia em tempo real via webcam com contagem por estabilidade

### Ambiente Controlado

As imagens sao capturadas em ambiente controlado: fundo escuro, iluminacao fixa e camera posicionada sobre uma rampa. Os itens deslizam pela rampa e sao detectados, classificados e contados automaticamente.

### Por que YOLOv8?

Diferente de classificadores tradicionais (como MobileNetV2), o YOLOv8 realiza **deteccao + classificacao em um unico passo**, fornecendo bounding boxes ao redor dos objetos. O modelo nano (YOLOv8n) tem apenas ~6MB e executa inferencia em ~55ms por frame na CPU, ideal para aplicacoes em tempo real.

## Passo 1 — Importacao das Bibliotecas

Utilizamos:
- **ultralytics**: framework YOLOv8 para deteccao de objetos
- **OpenCV**: data augmentation e captura de webcam
- **NumPy**: manipulacao de arrays e operacoes numericas
- **pathlib / os**: manipulacao de caminhos e diretorios

In [ ]:
import os
import random
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

print(f"OpenCV:      {cv2.__version__}")
print(f"NumPy:       {np.__version__}")
print("\nBibliotecas importadas com sucesso.")

## Passo 2 — Caminhos e Constantes

Definimos os diretorios do dataset, hiperparametros de treino e augmentation. As imagens base ficam em `dataset_base/` organizadas por categoria. O dataset aumentado e gerado em `dataset/`.

In [ ]:
BASE_DIR = Path(".").resolve()
DATASET_BASE_DIR = BASE_DIR / "dataset_base"
DATASET_DIR = BASE_DIR / "dataset"
DATA_YAML = BASE_DIR / "data.yaml"
RUNS_DIR = BASE_DIR / "runs"

CATEGORIES = ["arroz", "feijao", "outros"]
CLASS_MAP = {cat: i for i, cat in enumerate(CATEGORIES)}

# Augmentation
IMG_SIZE = 640
IMAGES_PER_INPUT = 10
SPLIT_RATIO = 0.2
SEED = 42

# Treino YOLO
YOLO_BASE_MODEL = "yolov8n.pt"
EPOCHS = 30
BATCH_SIZE = 8
TRAIN_NAME = "treino_alimentos"

# Inferencia
CONFIDENCE_THRESHOLD = 0.75
REQUIRED_STABLE_FRAMES = 10
COOLDOWN_SECONDS = 3.0

print(f"Categorias:     {CATEGORIES}")
print(f"IMG_SIZE:       {IMG_SIZE}")
print(f"Augmentacoes:   {IMAGES_PER_INPUT} por imagem")
print(f"EPOCHS:         {EPOCHS}")
print(f"BATCH_SIZE:     {BATCH_SIZE}")
print(f"Threshold:      {CONFIDENCE_THRESHOLD}")

print(f"\nDataset base:   {DATASET_BASE_DIR}")
for cat in CATEGORIES:
    cat_dir = DATASET_BASE_DIR / cat
    count = len(list(cat_dir.glob("*"))) if cat_dir.exists() else 0
    print(f"  {cat}: {count} imagens")

## Passo 3 — Data Augmentation com OpenCV

Para aumentar a diversidade do dataset e melhorar a robustez do modelo, aplicamos transformacoes aleatorias em cada imagem original:

| Tecnica | Parametros | Probabilidade |
|---------|-----------|---------------|
| Rotacao | ±20 graus | 70% |
| Flip | Horizontal, vertical, ambos | 50% |
| Brilho/Contraste | alpha [0.8, 1.2], beta [-30, 30] | 50% |
| Blur Gaussiano | kernel 3x3 ou 5x5 | 30% |
| Ruido Gaussiano | sigma = 10 | 30% |
| Zoom | escala [0.85, 1.15] | 50% |

Cada imagem gera 10 variantes aumentadas + a original redimensionada, totalizando ~825 imagens a partir de 75 originais.

In [ ]:
def adjust_brightness_contrast(image, alpha=1.0, beta=0):
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

def random_rotate(image):
    angle = np.random.uniform(-20, 20)
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REPLICATE)

def random_flip(image):
    flip_code = np.random.choice([-1, 0, 1])
    return cv2.flip(image, flip_code)

def random_blur(image):
    k = np.random.choice([3, 5])
    return cv2.GaussianBlur(image, (k, k), 0)

def random_noise(image):
    noise = np.random.normal(0, 10, image.shape).astype(np.uint8)
    return cv2.add(image, noise)

def random_zoom(image):
    h, w = image.shape[:2]
    scale = np.random.uniform(0.85, 1.15)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(image, (new_w, new_h))
    if scale > 1.0:
        start_x = (new_w - w) // 2
        start_y = (new_h - h) // 2
        return resized[start_y:start_y + h, start_x:start_x + w]
    else:
        canvas = np.zeros_like(image)
        start_x = (w - new_w) // 2
        start_y = (h - new_h) // 2
        canvas[start_y:start_y + new_h, start_x:start_x + new_w] = resized
        return canvas

def augment_image(image):
    aug = image.copy()
    if np.random.rand() < 0.7:
        aug = random_rotate(aug)
    if np.random.rand() < 0.5:
        aug = random_flip(aug)
    if np.random.rand() < 0.5:
        alpha = np.random.uniform(0.8, 1.2)
        beta = np.random.randint(-30, 30)
        aug = adjust_brightness_contrast(aug, alpha, beta)
    if np.random.rand() < 0.3:
        aug = random_blur(aug)
    if np.random.rand() < 0.3:
        aug = random_noise(aug)
    if np.random.rand() < 0.5:
        aug = random_zoom(aug)
    aug = cv2.resize(aug, (IMG_SIZE, IMG_SIZE))
    return aug

print("Funcoes de augmentation definidas.")

## Passo 4 — Gerar Dataset Aumentado

Aplica as augmentacoes em todas as imagens base e salva o resultado em `dataset/images/train/`. Cada imagem original gera a versao redimensionada (`_orig.jpg`) mais 10 variantes aumentadas (`_aug_0.jpg` a `_aug_9.jpg`).

In [ ]:
OUTPUT_DIR = DATASET_DIR / "images" / "train"
target_size = (IMG_SIZE, IMG_SIZE)
total_generated = 0

for category in sorted(os.listdir(DATASET_BASE_DIR)):
    input_category = os.path.join(DATASET_BASE_DIR, category)
    output_category = os.path.join(OUTPUT_DIR, category)
    os.makedirs(output_category, exist_ok=True)

    if not os.path.isdir(input_category):
        continue

    images = [f for f in os.listdir(input_category)
              if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]

    for img_name in images:
        img_path = os.path.join(input_category, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        image = cv2.resize(image, target_size)
        base_name = os.path.splitext(img_name)[0]

        # Salva original redimensionada
        cv2.imwrite(os.path.join(output_category, f"{base_name}_orig.jpg"), image)
        total_generated += 1

        # Gera variantes aumentadas
        for i in range(IMAGES_PER_INPUT):
            augmented = augment_image(image)
            cv2.imwrite(os.path.join(output_category, f"{base_name}_aug_{i}.jpg"), augmented)
            total_generated += 1

    count = len(os.listdir(output_category))
    print(f"  {category}: {len(images)} originais -> {count} imagens geradas")

print(f"\nTotal de imagens geradas: {total_generated}")

## Passo 5 — Dividir em Treino e Validacao

Separamos 20% das imagens para validacao. As imagens sao movidas de `dataset/images/train/` para `dataset/images/val/`, mantendo a estrutura de categorias.

In [ ]:
IMAGE_DIR = DATASET_DIR / "images" / "train"
VAL_DIR = DATASET_DIR / "images" / "val"

random.seed(SEED)

for category in os.listdir(IMAGE_DIR):
    category_path = os.path.join(IMAGE_DIR, category)
    val_category_path = os.path.join(VAL_DIR, category)

    if not os.path.isdir(category_path):
        continue

    os.makedirs(val_category_path, exist_ok=True)

    images = [f for f in os.listdir(category_path)
              if f.lower().endswith((".jpg", ".png", ".jpeg"))]

    random.shuffle(images)
    split_size = int(len(images) * SPLIT_RATIO)
    val_images = images[:split_size]

    for img in val_images:
        src = os.path.join(category_path, img)
        dst = os.path.join(val_category_path, img)
        shutil.move(src, dst)

    train_count = len(os.listdir(category_path))
    val_count = len(os.listdir(val_category_path))
    print(f"  {category}: {train_count} treino / {val_count} validacao")

print("\nDivisao train/val concluida.")

## Passo 6 — Gerar Labels YOLO

Para cada imagem, criamos um arquivo `.txt` no formato YOLO:
```
<classe> <x_centro> <y_centro> <largura> <altura>
```
Os valores sao normalizados entre 0 e 1. Como os itens sao centralizados na rampa (ambiente controlado), usamos bounding box fixa `0.5 0.5 0.8 0.8` — o objeto ocupa ~80% do frame centralizado.

In [ ]:
BASE_IMAGE_DIR = DATASET_DIR / "images"
BASE_LABEL_DIR = DATASET_DIR / "labels"

for split in ["train", "val"]:
    image_split_dir = os.path.join(BASE_IMAGE_DIR, split)
    label_split_dir = os.path.join(BASE_LABEL_DIR, split)
    os.makedirs(label_split_dir, exist_ok=True)

    for category in os.listdir(image_split_dir):
        img_folder = os.path.join(image_split_dir, category)
        label_folder = os.path.join(label_split_dir, category)

        if not os.path.isdir(img_folder):
            continue

        os.makedirs(label_folder, exist_ok=True)
        class_id = CLASS_MAP[category]

        for img in os.listdir(img_folder):
            if not img.lower().endswith((".jpg", ".png", ".jpeg")):
                continue
            name = os.path.splitext(img)[0]
            label_path = os.path.join(label_folder, name + ".txt")
            with open(label_path, "w") as f:
                f.write(f"{class_id} 0.5 0.5 0.8 0.8")

print("Labels gerados para train e val.")

## Passo 7 — Treinamento do YOLOv8

Utilizamos o **YOLOv8n** (nano) pre-treinado no COCO dataset como base. O modelo e retreinado com nosso dataset de alimentos por 30 epocas. O YOLOv8 gerencia automaticamente:
- Redimensionamento e normalizacao das imagens
- Augmentacoes internas adicionais
- Salvamento do melhor modelo por mAP
- Logs de treinamento e curvas de aprendizado

In [ ]:
model = YOLO(YOLO_BASE_MODEL)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name=TRAIN_NAME,
)

print(f"\nTreinamento concluido.")
print(f"Modelo salvo em: runs/detect/{TRAIN_NAME}/weights/best.pt")

## Passo 8 — Avaliacao do Modelo

Carregamos o melhor modelo treinado e executamos a validacao para obter as metricas:
- **mAP50**: mean Average Precision com IoU >= 0.50
- **mAP50-95**: mean Average Precision com IoU de 0.50 a 0.95
- **Precision e Recall** por classe

In [ ]:
weights_path = RUNS_DIR / "detect" / TRAIN_NAME / "weights" / "best.pt"

best_model = YOLO(str(weights_path))
val_results = best_model.val(data=str(DATA_YAML), imgsz=IMG_SIZE)

print("\n=== Resultados da Avaliacao ===")
print(f"mAP50:    {val_results.box.map50:.4f}")
print(f"mAP50-95: {val_results.box.map:.4f}")

class_names = best_model.names
for i, (p, r, ap50) in enumerate(
    zip(val_results.box.p, val_results.box.r, val_results.box.ap50)
):
    name = class_names.get(i, f"classe_{i}")
    print(f"  {name}: Precision={p:.4f}  Recall={r:.4f}  AP50={ap50:.4f}")

## Passo 9 — Inferencia em Tempo Real via Webcam

O sistema de inferencia usa **contagem por estabilidade**: o mesmo item precisa ser detectado por 10 frames consecutivos antes de ser contado. Isso evita contagens falsas causadas por deteccoes momentaneas. Apos contar, ha um cooldown de 3 segundos para evitar duplicatas.

### Como funciona
1. Cada frame e analisado pelo YOLOv8 — bounding boxes sao desenhadas automaticamente
2. A deteccao com maior confianca e selecionada
3. Se a mesma classe for detectada por 10 frames seguidos, o item e contado
4. Um painel mostra as contagens em tempo real
5. Pressione **q** para encerrar

In [ ]:
def run_webcam(model_path=None, camera_index=0):
    """Inferencia em tempo real via webcam com contagem por estabilidade."""
    if model_path is None:
        model_path = RUNS_DIR / "detect" / TRAIN_NAME / "weights" / "best.pt"

    if not Path(model_path).exists():
        print(f"Modelo nao encontrado em: {model_path}")
        return

    model = YOLO(str(model_path))
    cap = cv2.VideoCapture(camera_index)

    if not cap.isOpened():
        print(f"Nao foi possivel abrir a camera (indice {camera_index}).")
        return

    print(f"Camera iniciada (indice {camera_index}). Pressione 'q' para sair.\n")

    counts = {"arroz": 0, "feijao": 0, "outros": 0, "total": 0}
    stable_label = None
    stable_count = 0
    last_saved_label = None
    last_saved_time = 0.0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
        annotated = results[0].plot()

        current_label = None
        current_conf = 0.0

        if results[0].boxes is not None and len(results[0].boxes) > 0:
            best_box = max(results[0].boxes, key=lambda b: float(b.conf[0].item()))
            cls_id = int(best_box.cls[0].item())
            current_conf = float(best_box.conf[0].item())
            current_label = model.names[cls_id]

            if stable_label == current_label:
                stable_count += 1
            else:
                stable_label = current_label
                stable_count = 1

            if stable_count >= REQUIRED_STABLE_FRAMES:
                now = time.time()
                if (current_label != last_saved_label
                        or (now - last_saved_time) > COOLDOWN_SECONDS):
                    cat = current_label if current_label in counts else "outros"
                    counts[cat] += 1
                    counts["total"] += 1
                    last_saved_label = current_label
                    last_saved_time = now
                    stable_count = 0
                    print(f"[CONTADO] {current_label}  conf={current_conf:.2f}  total={counts['total']}")
        else:
            stable_label = None
            stable_count = 0

        # Painel de contagens
        if current_label:
            status = f"Detectando: {current_label} ({current_conf:.0%})"
            s_color = (0, 255, 0)
        else:
            status = "Nenhum objeto detectado"
            s_color = (0, 0, 255)

        cv2.rectangle(annotated, (5, 5), (280, 155), (30, 30, 30), -1)
        panel_lines = [
            f"Arroz:  {counts['arroz']}",
            f"Feijao: {counts['feijao']}",
            f"Outros: {counts['outros']}",
            f"Total:  {counts['total']}",
        ]
        for i, line in enumerate(panel_lines):
            color = (0, 255, 0) if i == 3 else (255, 255, 255)
            cv2.putText(annotated, line, (10, 30 + i * 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        cv2.putText(annotated, status, (10, 135),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, s_color, 1)

        # Barra de estabilidade
        if stable_label and stable_count < REQUIRED_STABLE_FRAMES:
            h_frame = annotated.shape[0]
            bar_y = h_frame - 30
            progress = stable_count / REQUIRED_STABLE_FRAMES
            bar_w = int(annotated.shape[1] * progress)
            cv2.rectangle(annotated, (0, bar_y), (annotated.shape[1], h_frame), (30, 30, 30), -1)
            cv2.rectangle(annotated, (0, bar_y), (bar_w, h_frame), (0, 200, 255), -1)
            cv2.putText(annotated, f"Estabilizando: {stable_count}/{REQUIRED_STABLE_FRAMES}",
                        (10, h_frame - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        cv2.imshow("YOLOv8 - Contagem de Alimentos (q para sair)", annotated)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

    print(f"\n=== Resumo da sessao ===")
    for k, v in counts.items():
        print(f"  {k}: {v}")


print("Funcao run_webcam() definida.")
print("Uso: run_webcam()  ou  run_webcam(camera_index=1)")

## Passo 10 — Executar Webcam

Descomente a linha abaixo para iniciar a camera. Use `camera_index=1` para webcam externa.

In [ ]:
# Descomente para iniciar a webcam:
# run_webcam()                  # camera padrao (indice 0)
# run_webcam(camera_index=1)    # webcam externa (indice 1)

## Conclusao

Neste notebook implementamos um pipeline completo de deteccao e contagem de embalagens de alimentos:

1. **Data Augmentation**: 6 tecnicas de augmentacao via OpenCV (rotacao, flip, brilho/contraste, blur, ruido, zoom) para expandir 75 imagens base para ~825
2. **YOLOv8n**: modelo nano pre-treinado, retreinado com nosso dataset — deteccao + classificacao em um unico passo
3. **Metricas**: mAP50 = 0.995, Precision = 0.988, Recall = 0.999
4. **Inferencia em tempo real**: contagem por estabilidade via webcam com painel visual
5. **Integracao com API**: o modelo treinado e usado pelo backend FastAPI para contagem automatica via endpoints REST e WebSocket

### Proximos passos planejados
- Salvar imagens de evidencia em AWS S3
- Frontend React com contagens em tempo real
- Suporte a mais categorias de alimentos